In [58]:
import os
from pdf2image import convert_from_path

pdf_path = "seccion_primera_19970530.pdf"
output_dir = "output_images"

# "pdf2image pillow"
# "sudo apt-get install poppler-utils"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Convert PDF pages to images
pages = convert_from_path(pdf_path)

for i, page in enumerate(pages, start=1):
    page_number = f"{i:03d}"
    path = os.path.join(output_dir, f"page_{page_number}.png")
    if os.path.exists(path):
        print(f"Page {page_number} already exists, skipping.")
        continue
    image_path = os.path.join(output_dir, f"page_{page_number}.png")
    page.save(image_path, "PNG")
    print(f"Saved {image_path}")

Page 001 already exists, skipping.
Page 002 already exists, skipping.
Page 003 already exists, skipping.
Page 004 already exists, skipping.
Page 005 already exists, skipping.
Page 006 already exists, skipping.
Page 007 already exists, skipping.
Page 008 already exists, skipping.
Page 009 already exists, skipping.
Page 010 already exists, skipping.
Page 011 already exists, skipping.
Page 012 already exists, skipping.
Page 013 already exists, skipping.
Page 014 already exists, skipping.
Page 015 already exists, skipping.
Page 016 already exists, skipping.
Page 017 already exists, skipping.
Page 018 already exists, skipping.
Page 019 already exists, skipping.
Page 020 already exists, skipping.
Page 021 already exists, skipping.
Page 022 already exists, skipping.
Page 023 already exists, skipping.
Page 024 already exists, skipping.
Page 025 already exists, skipping.
Page 026 already exists, skipping.
Page 027 already exists, skipping.
Page 028 already exists, skipping.


In [ ]:
import sqlite3
import json
from typing import Dict, Any, Optional, List

def create_response_table(db_path: str = "responses.db") -> None:
    """Create the responses table if it doesn't exist."""
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS responses (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            model TEXT NOT NULL,
            request_metadata TEXT,
            request_start_timestamp TEXT NOT NULL,
            request_end_timestamp TEXT NOT NULL,
            input_messages TEXT NOT NULL,
            response_json TEXT NOT NULL,
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    conn.commit()
    conn.close()

def store_response_json(response, messages: List[Dict], model: str, request_metadata: Dict[str, Any], request_start_timestamp: str, request_end_timestamp: str, db_path: str = "responses.db") -> int:
    """
    Store the OpenAI response JSON and input messages in SQLite database.
    
    Args:
        response: OpenAI response object with to_json() method
        messages: List of input messages sent to OpenAI
        model: The model name used for the response
        request_metadata: Metadata about the request, including image path and timestamps
        request_start_timestamp: Start timestamp for the request
        request_end_timestamp: End timestamp for the request
        db_path: Path to SQLite database file
    
    Returns:
        int: The ID of the inserted record
    """
    # Ensure table exists
    create_response_table(db_path)
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Convert response and messages to JSON strings
    response_json_str = response.to_json()
    messages_json_str = json.dumps(messages, ensure_ascii=False, indent=2)
    
    cursor.execute('''
        INSERT INTO responses (model, request_metadata, request_start_timestamp, request_end_timestamp, input_messages, response_json)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (model, json.dumps(request_metadata), request_start_timestamp, request_end_timestamp, messages_json_str, response_json_str))

    record_id = cursor.lastrowid
    conn.commit()
    conn.close()
    
    return record_id

def load_response_json(record_id: int, db_path: str = "responses.db") -> Optional[Dict[str, Any]]:
    """
    Load a response JSON and input messages from SQLite database by ID.
    
    Args:
        record_id: The ID of the record to load
        db_path: Path to SQLite database file
    
    Returns:
        Dict containing the parsed JSON response and input messages, or None if not found
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT input_messages, response_json FROM responses WHERE id = ?
    ''', (record_id,))
    
    result = cursor.fetchone()
    conn.close()
    
    if result:
        input_messages, response_json = result
        return {
            "input_messages": json.loads(input_messages),
            "response": json.loads(response_json)
        }
    return None

def load_latest_response_json(model: Optional[str] = None, db_path: str = "responses.db") -> Optional[Dict[str, Any]]:
    """
    Load the most recent response JSON and input messages from the database.
    
    Args:
        model: Optional model name to filter by
        db_path: Path to SQLite database file
    
    Returns:
        Dict containing the parsed JSON response and input messages, or None if not found
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    if model:
        cursor.execute('''
            SELECT input_messages, response_json FROM responses 
            WHERE model = ? 
            ORDER BY created_at DESC 
            LIMIT 1
        ''', (model,))
    else:
        cursor.execute('''
            SELECT input_messages, response_json FROM responses 
            ORDER BY created_at DESC 
            LIMIT 1
        ''')
    
    result = cursor.fetchone()
    conn.close()
    
    if result:
        input_messages, response_json = result
        return {
            "input_messages": json.loads(input_messages),
            "response": json.loads(response_json)
        }
    return None

def get_response_summary(record_id: int, db_path: str = "responses.db") -> Optional[Dict[str, Any]]:
    """
    Get a summary of the stored response including metadata.
    
    Args:
        record_id: The ID of the record to get summary for
        db_path: Path to SQLite database file

    Returns:
        Dict containing summary information, or None if not found
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT id, model, request_start_timestamp, request_end_timestamp, created_at FROM responses WHERE id = ?
    ''', (record_id,))

    result = cursor.fetchone()
    conn.close()

    if result:
        return {
            "id": result[0],
            "model": result[1],
            "request_start_timestamp": result[2],
            "request_end_timestamp": result[3],
            "created_at": result[4]
        }
    return None

def get_all_responses(db_path: str = "responses.db") -> List[Dict[str, Any]]:
    """
    Get all stored responses with metadata.

    Args:
        db_path: Path to SQLite database file

    Returns:
        List of dictionaries containing response metadata
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute('''
        SELECT * FROM responses
    ''')

    results = cursor.fetchall()
    conn.close()

    return results

In [64]:
import base64
from openai import OpenAI

from dotenv import load_dotenv

import datetime

from pathlib import Path

load_dotenv()

client = OpenAI()

BASE_PATH = Path.cwd()
IMAGES_PATH = BASE_PATH.joinpath("output_images")

In [65]:
# models = client.models.list()
# models.to_dict()

In [66]:
import re

# To get the last match specifically
def prepare_iteration_instructions_message(text):
    # Regex pattern to match ARTÍCULO followed by a number and °
    pattern = r'ARTÍCULO\s+(\d+)°\s*—\s*(.*?)(?=\n\n|\nARTÍCULO|\Z)'
    matches = re.findall(pattern, text, re.DOTALL)
    message = ""
    if matches:
        article_number, article_content = matches[-1]
        message += f"The last extracted article is: ARTÍCULO {article_number}° with text {article_content.strip()}."
    if text.endswith("This is the end of the norm extraction."):
        message += " The norm extraction is complete."
    else:
        message += " The norm extraction continues in the next image. This article might be incomplete, please check the next image for more articles."
    return message if message else "No articles found or the norm extraction is incomplete."

# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

# Getting the Base64 string
base64_image = encode_image(image_path)

# | model        | input | output | url                                                  |
# | ------------ | ----- | ------ | ---------------------------------------------------- |
# | o4-mini      | 1.10  | 4.40   | https://platform.openai.com/docs/models/o4-mini      |
# | gpt-4.1-nano | 0.10  | 0.40   | https://platform.openai.com/docs/models/gpt-4.1-nano |
# | gpt-4o-mini  | 0.15  | 0.60   | https://platform.openai.com/docs/models/gpt-4o-mini  |
# | gpt-4.1-mini | 0.40  | 1.60   | https://platform.openai.com/docs/models/gpt-4.1-mini |

# selected_model = "gpt-4.1-nano"
# selected_model = "gpt-4o-mini"
# selected_model = "gpt-4.1-mini"
selected_model = "o4-mini" # is the model that worked the best using reasoning.

norm_to_extract = "CODIGO DE MINERIA DE LA REPUBLICA ARGENTINA"

default_system_message = {
    "role": "developer",
    "content": f"""
You are a data entry expert that has to extract data from the "Boletines Oficiales de la República Argentina"
That is newspaper from a specific date that contain legal texts.
You will receive a series of images.
For the image provided, please extract the text related to the "{norm_to_extract}"

The output shall be in markdown format with a structure like this:

```md
# <norm title>

## TÍTULO PRIMERO - <text of title 1>

ARTÍCULO <number of the article n>° — <text of article n>

### CAPÍTULO I - <text of capitulo 1>

ARTÍCULO <n>° — <text of article n>

## TÍTULO SEGUNDO - <text of title 2>
```

Have the following considerations in mind:
- The text can be structured in different text columns in the image, like in a newspaper.
- The "CAPÍTULOS" can be denoted by characters like "§" or "CAPÍTULO", for example: § I. If the notation `§ I` is inside a "CAPITULO", then it corresponds to a "SECCIÓN" of that "CAPÍTULO".
    Overall the structure shall be:
    ```md
    Libro
    └── Título
        └── Capítulo
            └── Sección
                    └── Artículo
                        └── Inciso
                            └── Párrafo / Apartado
    ```
- Extend the markdown structure to include all titles, articles, and chapters as they appear in the document.
- If the norm continues, add the following text at the end of the markdown:
```md
---
This norm continues in the next image.
```
- If the norm is complete, add the following text at the end of the markdown:
```md
---
This is the end of the norm extraction.
```
- exclude from the response any extra text or tags like: ```md or ```
""",
}

In [ ]:
response = None

for i, image_path in enumerate(IMAGES_PATH.glob("*.png"), start=1):
    print(f"Processing image: {image_path}")
    # Getting the Base64 string
    base64_image = encode_image(image_path)

    if i == 1:
        input_message = [
            default_system_message,
            {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
            }
        ]
    else:
        if response.output_text.endswith("This is the end of the norm extraction."):
            print("Norm extraction is complete, no more images will be processed.")
            break
        input_message = [
            default_system_message,
            {
                "role": "user",
                "content": prepare_iteration_instructions_message(base64_image),
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                    },
                ],
            }
        ]

    start_timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
    response = client.responses.create(
        model=selected_model,
        input=input_message,
    )
    end_timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()

    store_response_json(
        response=response,
        request_metadata={
            "image_path": image_path,
            "request_start_timestamp": start_timestamp,
            "request_end_timestamp": end_timestamp
        },
        messages=input_message,
        model=selected_model,
        request_start_timestamp=start_timestamp,
        request_end_timestamp=end_timestamp
    )

# with open(f"{date}_output_{selected_model}.md", mode="a", encoding="utf-8") as f:
#     f.write(response.output_text)

In [ ]:
get_all_responses()

In [40]:
import json
conn = sqlite3.connect("responses.db")
cursor = conn.cursor()

cursor.execute('''
    SELECT * FROM responses
''')

results = cursor.fetchall()
conn.close()


content = []

# Sort results by created_at in descending order
results.sort(key=lambda x: x[6], reverse=False)

for result in results:
    id__, model, request_start_timestamp, request_end_timestamp, input_messages, response_json, created_at = result
    data = json.loads(response_json)["output"]
    content.append([item["content"][0]["text"] for item in data if item["type"] == "message" and "content" in item.keys()][0])

content = "\n".join(content)
content = content.replace("\n---\nThis norm continues in the next image.", "").strip()
content = content.replace("\n\n# CODIGO DE MINERIA DE LA REPUBLICA ARGENTINA", "").strip()
content = content.replace("---\nThis is the end of the norm extraction.", "").strip()

with open("output_codigo_mineria.md", "w", encoding="utf-8") as f:
    f.write(content)

content

"# CODIGO DE MINERIA DE LA REPUBLICA ARGENTINA\n\n## TÍTULO PRIMERO - DE LAS MINAS Y SU DOMINIO\n\nARTÍCULO 1° — El Código de Minería rige los derechos, obligaciones y procedimientos referentes a la adquisición, exploración y aprovechamiento de las sustancias minerales.\n\n### CAPÍTULO I - Clasificación y división de las minas (§ I)\n\nARTÍCULO 2° — Con relación a los derechos que este Código reconoce y asegura, las minas se dividen en tres categorías:  \n1° Minas de las que el suelo es un accesorio, y que pertenecen exclusivamente al Estado, y que sólo pueden explotarse en virtud de concesión legal otorgada por autoridad competente.  \n2° Minas que, por razón de su importancia, se conceden preferentemente al dueño del suelo; y minas que, por las condiciones de su yacimiento, se destinan al aprovechamiento común.  \n3° Minas que pertenecen únicamente al propietario, y que nadie puede explotar sin su consentimiento, salvo por motivo de utilidad pública.\n\nARTÍCULO 3° — Corresponden a l

"# CODIGO DE MINERIA DE LA REPUBLICA ARGENTINA\n\n## TÍTULO PRIMERO - DE LAS MINAS Y SU DOMINIO\n\nARTÍCULO 1° — El Código de Minería rige los derechos, obligaciones y procedimientos referentes a la adquisición, exploración y aprovechamiento de las sustancias minerales.\n\n### CAPÍTULO I - Clasificación y división de las minas (§ I)\n\nARTÍCULO 2° — Con relación a los derechos que este Código reconoce y asegura, las minas se dividen en tres categorías:  \n1° Minas de las que el suelo es un accesorio, y que pertenecen exclusivamente al Estado, y que sólo pueden explotarse en virtud de concesión legal otorgada por autoridad competente.  \n2° Minas que, por razón de su importancia, se conceden preferentemente al dueño del suelo; y minas que, por las condiciones de su yacimiento, se destinan al aprovechamiento común.  \n3° Minas que pertenecen únicamente al propietario, y que nadie puede explotar sin su consentimiento, salvo por motivo de utilidad pública.\n\nARTÍCULO 3° — Corresponden a l

In [25]:
import json

content = []

for result in results:
    id__, model, request_start_timestamp, request_end_timestamp, input_messages, response_json, created_at = result
    data = json.loads(response_json)["output"]
    content.append([item["content"][0]["text"] for item in data if item["type"] == "message" and "content" in item.keys()][0])

content = "\n".join(content)
content = content.replace("---\nThis norm continues in the next image.# CÓDIGO DE MINERÍA DE LA REPÚBLICA ARGENTINA", "").strip()
content = content.replace("```md", "").replace("```", "").strip()
content = content.replace("This norm continues in the next image.---", "").strip()
content = content.replace("--- ", "").strip()
content = content.replace("This is the end of the norm extraction.", "").strip()
content = content.replace("This norm continues in the next image.", "").strip()
content = content.replace("CÓDIGO DE MINERÍA DE LA REPÚBLICA ARGENTINA", "").strip()

with open("output_codigo_mineria.md", "w", encoding="utf-8") as f:
    f.write(content)

with open("output_codigo_mineria.md", "r", encoding="utf-8") as f:
    print(f.read())

# CODIGO DE MINERIA DE LA REPUBLICA ARGENTINA

## TÍTULO PRIMERO - DE LAS MINAS Y SU DOMINIO

ARTÍCULO 1° — El Código de Minería rige los derechos, obligaciones y procedimientos referentes a la adquisición, exploración y aprovechamiento de las sustancias minerales.

### CAPÍTULO I - Clasificación y división de las minas (§ I)

ARTÍCULO 2° — Con relación a los derechos que este Código reconoce y asegura, las minas se dividen en tres categorías:  
1° Minas de las que el suelo es un accesorio, y que pertenecen exclusivamente al Estado, y que sólo pueden explotarse en virtud de concesión legal otorgada por autoridad competente.  
2° Minas que, por razón de su importancia, se conceden preferentemente al dueño del suelo; y minas que, por las condiciones de su yacimiento, se destinan al aprovechamiento común.  
3° Minas que pertenecen únicamente al propietario, y que nadie puede explotar sin su consentimiento, salvo por motivo de utilidad pública.

ARTÍCULO 3° — Corresponden a la primera cate

In [23]:
with open("output_codigo_mineria.md", "r", encoding="utf-8") as f:
    print(f.read())

# CODIGO DE MINERIA DE LA REPUBLICA ARGENTINA

## TÍTULO PRIMERO - DE LAS MINAS Y SU DOMINIO

ARTÍCULO 1° — El Código de Minería rige los derechos, obligaciones y procedimientos referentes a la adquisición, exploración y aprovechamiento de las sustancias minerales.

### CAPÍTULO I - Clasificación y división de las minas (§ I)

ARTÍCULO 2° — Con relación a los derechos que este Código reconoce y asegura, las minas se dividen en tres categorías:  
1° Minas de las que el suelo es un accesorio, y que pertenecen exclusivamente al Estado, y que sólo pueden explotarse en virtud de concesión legal otorgada por autoridad competente.  
2° Minas que, por razón de su importancia, se conceden preferentemente al dueño del suelo; y minas que, por las condiciones de su yacimiento, se destinan al aprovechamiento común.  
3° Minas que pertenecen únicamente al propietario, y que nadie puede explotar sin su consentimiento, salvo por motivo de utilidad pública.

ARTÍCULO 3° — Corresponden a la primera cate

In [ ]:
with open("output_codigo_mineria.md", "w", encoding="utf-8") as f:

    content = [item["content"][0]["text"] for item in data if item["type"] == "message" and "content" in item.keys()][0]
    content = content.replace("---\nThis norm continues in the next image.# CÓDIGO DE MINERÍA DE LA REPÚBLICA ARGENTINA", "").strip()
    content = content.replace("```md", "").replace("```", "").strip()
    content = content.replace("This norm continues in the next image.---", "").strip()
    content = content.replace("--- ", "").strip()
    content = content.replace("This is the end of the norm extraction.", "").strip()
    content = content.replace("This norm continues in the next image.", "").strip()
    content = content.replace("CÓDIGO DE MINERÍA DE LA REPÚBLICA ARGENTINA", "").strip()

    f.write(content)
